**MODULE-2 ASSIGNMENT- Independent Multi-Omics Target Discovery: Lung Adenocarcinoma (LUAD)**


integrating CPTAC transcriptomics and proteomics with GWAS Catalog genomic data to identify and rank candidate disease targetS FOR LUAD

**Part A — Build the RNA + protein files (CPTAC LUAD, tumor vs. normal)**

In [ ]:
help(cptac.download)

Help on function download in module cptac.tools.download_tools:

download(cancer: str, source: str, dtype: str, data_file: str) -> bool
    Downloads data files for a specific cancer, source, datatype, and file name from Zenodo
    
    :param cancer: The cancer type (e.g. 'brca').
    :param source: The data source (e.g. 'harmonized').
    :param dtype: The datatype of the files to download (e.g. 'clinical')
    :param data_file: The file name to download (look at index.tsv for examples).
    
    :return: True if data has successfully downloaded; raises error otherwise.



In [ ]:
import cptac
import pandas as pd
import numpy as np
from scipy import stats

luad = cptac.Luad()
luad.list_data_sources()

,Data type,Available sources
0,CNV,"[bcm, washu]"
1,circular_RNA,[bcm]
2,miRNA,"[bcm, washu]"
3,phosphoproteomics,"[bcm, umich]"
4,proteomics,"[bcm, umich]"
5,transcriptomics,"[bcm, broad, washu]"
6,ancestry_prediction,[harmonized]
7,somatic_mutation,"[harmonized, washu]"
8,clinical,[mssm]
9,follow-up,[mssm]


In [ ]:
rna = luad.get_transcriptomics('bcm')
protein = luad.get_proteomics('umich')
clinical = luad.get_clinical('mssm')

print("RNA shape:", rna.shape)
print("Protein shape:", protein.shape)
print("Clinical shape:", clinical.shape)
print()
print("Sample IDs (first 10):", rna.index[:10].tolist())

/opt/anaconda3/envs/biot6900/lib/python3.11/site-packages/cptac/cancers/bcm/bcmluad.py:143: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  transcript = transcript.reset_index()
/opt/anaconda3/envs/biot6900/lib/python3.11/site-packages/cptac/cancers/umich/umichluad.py:145: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  df['Database_ID'] = df.Index.apply(lambda x: x.split('|')[0]) # get protein identifier
/opt/anaconda3/envs/biot6900/lib/python3.11/site-packages/cptac/cancers/umich/umichluad.py:146: PerformanceWarning: DataFrame is 

RNA shape: (213, 59286)
Protein shape: (213, 13302)
Clinical shape: (111, 124)

Sample IDs (first 10): ['11LU013', '11LU016', '11LU022', '11LU035', 'C3L-00001', 'C3L-00009', 'C3L-00080', 'C3L-00083', 'C3L-00093', 'C3L-00094']


In [ ]:
normal_like = [s for s in rna.index if s.endswith('.N')]
print(f"Samples ending in .N: {len(normal_like)}")
print(normal_like[:10])

Samples ending in .N: 102
['C3L-00001.N', 'C3L-00009.N', 'C3L-00080.N', 'C3L-00083.N', 'C3L-00093.N', 'C3L-00094.N', 'C3L-00095.N', 'C3L-00140.N', 'C3L-00144.N', 'C3L-00263.N']


In [ ]:
tumor_ids = [s for s in rna.index if not s.endswith('.N')]
normal_ids = [s for s in rna.index if s.endswith('.N')]

print(f"Tumor samples: {len(tumor_ids)}")
print(f"Normal samples: {len(normal_ids)}")

Tumor samples: 111
Normal samples: 102


In [ ]:
def compute_de(data, tumor_ids, normal_ids):
    rows = []
    for gene in data.columns:
        t_vals = data.loc[tumor_ids, gene].dropna()
        n_vals = data.loc[normal_ids, gene].dropna()
        if len(t_vals) > 5 and len(n_vals) > 5:
            log2fc = t_vals.mean() - n_vals.mean()
            _, pval = stats.ttest_ind(t_vals, n_vals, nan_policy='omit')
            rows.append({'gene': gene, 'log2fc': log2fc, 'pval': pval})
    return pd.DataFrame(rows)

rna_de = compute_de(rna, tumor_ids, normal_ids)
print(rna_de.shape)
rna_de.head()

(59286, 3)


,gene,log2fc,pval
0,"(A1BG, ENSG00000121410.12)",-0.032417,7.322520e-01
1,"(A1BG-AS1, ENSG00000268895.6)",-0.381007,4.045847e-05
2,"(A1CF, ENSG00000148584.15)",0.930620,9.413446e-07
3,"(A2M, ENSG00000175899.15)",-1.953055,1.210789e-58
4,"(A2M-AS1, ENSG00000245105.4)",-1.363206,1.210617e-41


In [ ]:
rna_de['gene'] = rna_de['gene'].apply(lambda x: x[0] if isinstance(x, tuple) else x)
rna_de.head()

,gene,log2fc,pval
0,A1BG,-0.032417,7.322520e-01
1,A1BG-AS1,-0.381007,4.045847e-05
2,A1CF,0.930620,9.413446e-07
3,A2M,-1.953055,1.210789e-58
4,A2M-AS1,-1.363206,1.210617e-41


In [ ]:
protein_de = compute_de(protein, tumor_ids, normal_ids)
protein_de['gene'] = protein_de['gene'].apply(lambda x: x[0] if isinstance(x, tuple) else x)

print(protein_de.shape)
protein_de.head()

(12900, 3)


,gene,log2fc,pval
0,ARF5,0.381317,5.933426e-27
1,M6PR,0.165549,4.908744e-06
2,ESRRA,-0.218910,2.000926e-03
3,FKBP4,0.539501,2.185516e-26
4,NDUFAF7,-0.044908,1.902181e-01


In [ ]:
import os
os.makedirs('data', exist_ok=True)

rna_de.to_csv('data/luad_transcriptomics.tsv', sep='\t', index=False)
protein_de.to_csv('data/luad_proteomics.tsv', sep='\t', index=False)

print("Saved both files to data/")

Saved both files to data/


In [ ]:
import os
print(os.listdir('data'))

['cptac_brca_protein.tsv', 'luad_transcriptomics.tsv', 'luad_proteomics.tsv', 'cptac_brca_mutation.tsv', 'cptac_brca_rna.tsv']


In [ ]:
import os
print(os.listdir('data'))

['cptac_brca_protein.tsv', 'luad_transcriptomics.tsv', 'luad_proteomics.tsv', 'cptac_brca_mutation.tsv', 'cptac_brca_rna.tsv']


**Part B- Build the GWAS file**

Downloaded the gene trait data of lung adenocarcinoma from https://www.ebi.ac.uk/gwas/  

In [ ]:
gwas_raw = pd.read_csv('gwas-association-downloaded_2026-09-24-MONDO_0005061.tsv', sep='\t')
print(gwas_raw.columns.tolist())
print(gwas_raw.shape)
gwas_raw.head()

['DATE ADDED TO CATALOG', 'PUBMEDID', 'FIRST AUTHOR', 'DATE', 'JOURNAL', 'LINK', 'STUDY', 'DISEASE/TRAIT', 'INITIAL SAMPLE SIZE', 'REPLICATION SAMPLE SIZE', 'REGION', 'CHR_ID', 'CHR_POS', 'REPORTED GENE(S)', 'MAPPED_GENE', 'UPSTREAM_GENE_ID', 'DOWNSTREAM_GENE_ID', 'SNP_GENE_IDS', 'UPSTREAM_GENE_DISTANCE', 'DOWNSTREAM_GENE_DISTANCE', 'STRONGEST SNP-RISK ALLELE', 'SNPS', 'MERGED', 'SNP_ID_CURRENT', 'CONTEXT', 'INTERGENIC', 'RISK ALLELE FREQUENCY', 'P-VALUE', 'PVALUE_MLOG', 'P-VALUE (TEXT)', 'OR or BETA', '95% CI (TEXT)', 'PLATFORM [SNPS PASSING QC]', 'CNV', 'MAPPED_TRAIT', 'MAPPED_TRAIT_URI', 'STUDY ACCESSION', 'GENOTYPING TECHNOLOGY']
(2833, 38)


,DATE ADDED TO CATALOG,PUBMEDID,FIRST AUTHOR,DATE,JOURNAL,LINK,STUDY,DISEASE/TRAIT,INITIAL SAMPLE SIZE,REPLICATION SAMPLE SIZE,...,PVALUE_MLOG,P-VALUE (TEXT),OR or BETA,95% CI (TEXT),PLATFORM [SNPS PASSING QC],CNV,MAPPED_TRAIT,MAPPED_TRAIT_URI,STUDY ACCESSION,GENOTYPING TECHNOLOGY
0,2025-07-14,37167549,Shen S,2023-05-11,Am J Respir Crit Care Med,www.ncbi.nlm.nih.gov/pubmed/37167549,A Large-Scale Exome-Wide Association Study Ide...,Adenocarcinoma,"1,359 European ancestry cases, 334,643 Europea...","10,113 European ancestry cases, 318,259 Europe...",...,5.698970,NaN,2.008,[0.74-3.28] unit increase,Illumina [216739] (imputed),N,lung adenocarcinoma,http://purl.obolibrary.org/obo/MONDO_0005061,GCST90296661,Genome-wide genotyping array
1,2025-07-14,37167549,Shen S,2023-05-11,Am J Respir Crit Care Med,www.ncbi.nlm.nih.gov/pubmed/37167549,A Large-Scale Exome-Wide Association Study Ide...,Adenocarcinoma,"1,359 European ancestry cases, 334,643 Europea...","10,113 European ancestry cases, 318,259 Europe...",...,5.221849,NaN,11.939,[6.78-17.1] unit increase,Illumina [216739] (imputed),N,lung adenocarcinoma,http://purl.obolibrary.org/obo/MONDO_0005061,GCST90296661,Genome-wide genotyping array
2,2025-07-14,37167549,Shen S,2023-05-11,Am J Respir Crit Care Med,www.ncbi.nlm.nih.gov/pubmed/37167549,A Large-Scale Exome-Wide Association Study Ide...,Adenocarcinoma,"1,359 European ancestry cases, 334,643 Europea...","10,113 European ancestry cases, 318,259 Europe...",...,6.045757,NaN,0.821,[0.2-1.44] unit increase,Illumina [216739] (imputed),N,lung adenocarcinoma,http://purl.obolibrary.org/obo/MONDO_0005061,GCST90296661,Genome-wide genotyping array
3,2025-07-14,37167549,Shen S,2023-05-11,Am J Respir Crit Care Med,www.ncbi.nlm.nih.gov/pubmed/37167549,A Large-Scale Exome-Wide Association Study Ide...,Adenocarcinoma,"1,359 European ancestry cases, 334,643 Europea...","10,113 European ancestry cases, 318,259 Europe...",...,5.698970,NaN,2.139,[0.46-3.82] unit increase,Illumina [216739] (imputed),N,lung adenocarcinoma,http://purl.obolibrary.org/obo/MONDO_0005061,GCST90296661,Genome-wide genotyping array
4,2025-07-14,37167549,Shen S,2023-05-11,Am J Respir Crit Care Med,www.ncbi.nlm.nih.gov/pubmed/37167549,A Large-Scale Exome-Wide Association Study Ide...,Adenocarcinoma,"1,359 European ancestry cases, 334,643 Europea...","10,113 European ancestry cases, 318,259 Europe...",...,5.698970,NaN,0.121,[0.076-0.166] unit increase,Illumina [216739] (imputed),N,lung adenocarcinoma,http://purl.obolibrary.org/obo/MONDO_0005061,GCST90296661,Genome-wide genotyping array


In [ ]:
print(gwas_raw.columns.tolist())

['DATE ADDED TO CATALOG', 'PUBMEDID', 'FIRST AUTHOR', 'DATE', 'JOURNAL', 'LINK', 'STUDY', 'DISEASE/TRAIT', 'INITIAL SAMPLE SIZE', 'REPLICATION SAMPLE SIZE', 'REGION', 'CHR_ID', 'CHR_POS', 'REPORTED GENE(S)', 'MAPPED_GENE', 'UPSTREAM_GENE_ID', 'DOWNSTREAM_GENE_ID', 'SNP_GENE_IDS', 'UPSTREAM_GENE_DISTANCE', 'DOWNSTREAM_GENE_DISTANCE', 'STRONGEST SNP-RISK ALLELE', 'SNPS', 'MERGED', 'SNP_ID_CURRENT', 'CONTEXT', 'INTERGENIC', 'RISK ALLELE FREQUENCY', 'P-VALUE', 'PVALUE_MLOG', 'P-VALUE (TEXT)', 'OR or BETA', '95% CI (TEXT)', 'PLATFORM [SNPS PASSING QC]', 'CNV', 'MAPPED_TRAIT', 'MAPPED_TRAIT_URI', 'STUDY ACCESSION', 'GENOTYPING TECHNOLOGY']


In [ ]:
gwas_raw['gene'] = gwas_raw['MAPPED_GENE'].astype(str).str.split(', ')
gwas_exploded = gwas_raw.explode('gene')

gwas_exploded = gwas_exploded[gwas_exploded['gene'].notna() & (gwas_exploded['gene'] != 'nan')]

gwas_gene_level = gwas_exploded.groupby('gene')['PVALUE_MLOG'].max().reset_index()
gwas_gene_level.columns = ['gene', 'neglog10p']

print(gwas_gene_level.shape)
gwas_gene_level.head()

(275, 2)


,gene,neglog10p
0,ACTR2,16.397940
1,ACTR2 - SPRED2,11.522879
2,ACTRT3 - MYNN,11.698970
3,ACVR1B,8.698970
4,ADAMTS7,14.397940


In [ ]:
import re

def safe_split(x):
    if pd.isna(x):
        return []
    return re.split(r',\s*|\s+-\s+', str(x))

gwas_raw['gene'] = gwas_raw['MAPPED_GENE'].apply(safe_split)
gwas_exploded = gwas_raw.explode('gene')
gwas_exploded['gene'] = gwas_exploded['gene'].astype(str).str.strip()

gwas_exploded = gwas_exploded[gwas_exploded['gene'].notna() & (gwas_exploded['gene'] != 'nan') & (gwas_exploded['gene'] != '')]

gwas_gene_level = gwas_exploded.groupby('gene')['PVALUE_MLOG'].max().reset_index()
gwas_gene_level.columns = ['gene', 'neglog10p']

print(gwas_gene_level.shape)
gwas_gene_level.head(10)

(361, 2)


,gene,neglog10p
0,ACTR2,16.397940
1,ACTRT3,11.698970
2,ACVR1B,8.698970
3,ADAMTS7,14.397940
4,AJAP1,5.221849
5,AK5,15.397940
6,AK6P1,5.301030
7,AKR1B10,5.522879
8,ANP32A,5.154902
9,APOM,11.301030


In [ ]:
gwas_gene_level.to_csv('data/luad_gwas.tsv', sep='\t', index=False)
print("Saved!")

Saved!


In [ ]:
import os
print(os.listdir('data'))

['cptac_brca_protein.tsv', 'luad_transcriptomics.tsv', 'luad_gwas.tsv', 'luad_proteomics.tsv', 'cptac_brca_mutation.tsv', 'cptac_brca_rna.tsv']


**PART C- HARMONIZE/CONCORDANCE/SCORE/RANKING**

In [ ]:
tx = pd.read_csv('data/luad_transcriptomics.tsv', sep='\t')
pr = pd.read_csv('data/luad_proteomics.tsv', sep='\t')
gw = pd.read_csv('data/luad_gwas.tsv', sep='\t')

print(tx.columns.tolist(), tx.shape)
print(pr.columns.tolist(), pr.shape)
print(gw.columns.tolist(), gw.shape)

['gene', 'log2fc', 'pval'] (59286, 3)
['gene', 'log2fc', 'pval'] (12900, 3)
['gene', 'neglog10p'] (361, 2)


In [ ]:
tx_renamed = tx.rename(columns={"log2fc": "rna_lfc", "pval": "rna_p"})
pr_renamed = pr.rename(columns={"log2fc": "prot_lfc", "pval": "prot_p"})

df = tx_renamed.merge(pr_renamed, on="gene", how="inner").merge(gw, on="gene", how="inner")
print(f"Genes surviving the 3-way join: {len(df)}")
df.head()

Genes surviving the 3-way join: 154


,gene,rna_lfc,rna_p,prot_lfc,prot_p,neglog10p
0,ACTR2,0.178249,2.049623e-04,-0.065713,6.142864e-05,16.397940
1,ACVR1B,0.573731,5.440020e-12,0.230559,1.784861e-02,8.698970
2,ADAMTS7,-0.398105,2.400133e-05,0.622743,1.623066e-11,14.397940
3,AKR1B10,2.808574,2.835966e-14,0.293399,2.023843e-01,5.522879
4,ANP32A,-0.195578,5.809513e-06,0.265140,8.506599e-12,5.154902


In [ ]:
import numpy as np

df["concordant"] = np.sign(df["rna_lfc"]) == np.sign(df["prot_lfc"])
print(f"Sign-concordant genes: {df['concordant'].sum()} / {len(df)}")

Sign-concordant genes: 112 / 154


In [ ]:
def rank_percentile(series: pd.Series) -> pd.Series:
    """Normalize any score to [0, 1] by rank. Robust to outliers and scale differences."""
    return series.rank(method="average", pct=True)


def multi_evidence_score(df: pd.DataFrame, cols, weights) -> pd.Series:
    """Weighted sum of rank-percentile-normalized layer scores."""
    normed = pd.DataFrame({c: rank_percentile(df[c]) for c in cols})
    return sum(weights[c] * normed[c] for c in cols)

In [ ]:
EQUAL_WEIGHTS = {"transcriptomic": 1/3, "proteomic": 1/3, "genomic": 1/3}

In [ ]:
df["transcriptomic"] = df["rna_lfc"].abs()
df["proteomic"] = df["prot_lfc"].abs()
df["genomic"] = df["neglog10p"]

df["score"] = multi_evidence_score(df, ["transcriptomic", "proteomic", "genomic"], EQUAL_WEIGHTS)

In [ ]:
df[["gene", "transcriptomic", "proteomic", "genomic", "score"]].head()

,gene,transcriptomic,proteomic,genomic,score
0,ACTR2,0.178249,0.065713,16.397940,0.417749
1,ACVR1B,0.573731,0.230559,8.698970,0.569264
2,ADAMTS7,0.398105,0.622743,14.397940,0.742424
3,AKR1B10,2.808574,0.293399,5.522879,0.596320
4,ANP32A,0.195578,0.265140,5.154902,0.287879


In [ ]:
ranked = df.sort_values("score", ascending=False)
top = ranked.head(15)

ranked.to_csv("targets_luad.csv", index=False)

print(top[["gene", "rna_lfc", "prot_lfc", "neglog10p", "concordant", "score"]].round(3).to_string(index=False))

   gene  rna_lfc  prot_lfc  neglog10p  concordant  score
  FGFR2   -1.718    -0.734     34.097        True  0.944
   KRT8    1.472     0.639     14.699        True  0.883
 USHBP1   -2.370    -1.277      8.699        True  0.881
  GIPC2   -2.001    -1.134      7.398        True  0.847
CLPTM1L    1.063     0.468     28.000        True  0.842
  BRCA2    0.958    -0.667     11.097       False  0.820
 DNAJB4   -1.375    -0.795      7.398        True  0.817
 CYP1A1   -2.523    -1.026      6.301        True  0.813
   PALM   -1.014    -0.970      8.523        True  0.813
  EPHX2   -0.836    -0.653     11.222        True  0.807
    FRY   -0.929    -0.680      8.398        True  0.781
SLC12A7    1.104     0.564      7.000        True  0.750
    VWF   -2.510    -1.290      5.523        True  0.746
ADAMTS7   -0.398     0.623     14.398       False  0.742
  FADS2    0.638     0.453     13.699        True  0.739


In [ ]:
ranked = df.sort_values("score", ascending=False)
top = ranked.head(15)

ranked.to_csv("targets_luad.csv", index=False)

print(top[["gene", "rna_lfc", "prot_lfc", "neglog10p", "concordant", "score"]].round(3).to_string(index=False))

   gene  rna_lfc  prot_lfc  neglog10p  concordant  score
  FGFR2   -1.718    -0.734     34.097        True  0.944
   KRT8    1.472     0.639     14.699        True  0.883
 USHBP1   -2.370    -1.277      8.699        True  0.881
  GIPC2   -2.001    -1.134      7.398        True  0.847
CLPTM1L    1.063     0.468     28.000        True  0.842
  BRCA2    0.958    -0.667     11.097       False  0.820
 DNAJB4   -1.375    -0.795      7.398        True  0.817
 CYP1A1   -2.523    -1.026      6.301        True  0.813
   PALM   -1.014    -0.970      8.523        True  0.813
  EPHX2   -0.836    -0.653     11.222        True  0.807
    FRY   -0.929    -0.680      8.398        True  0.781
SLC12A7    1.104     0.564      7.000        True  0.750
    VWF   -2.510    -1.290      5.523        True  0.746
ADAMTS7   -0.398     0.623     14.398       False  0.742
  FADS2    0.638     0.453     13.699        True  0.739


In [ ]:
import os
print(os.path.exists('targets_luad.csv'))
print(os.listdir('.'))

True
['BIOT6900_Module2_Starter (1).ipynb', 'gwas-association-downloaded_2026-09-24-MONDO_0005061.tsv', '.DS_Store', 'targets_luad.csv', 'module1_setup.ipynb', '.claude', 'README.md', 'Interpretation.txt', '.ipynb_checkpoints', '.git', 'Data']
